In [ ]:
! pip install kaggle

**Importing Dependencies**

In [ ]:
!pip install tensorflow
import tensorflow as tf

In [3]:
import os
import json
from zipfile import ZipFile
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


**Data Collection**

In [4]:
kaggle_dictionary = json.load(open('/content/kaggle.json'))

In [5]:
kaggle_dictionary.keys()

dict_keys(['username', 'key'])

In [6]:
# Setup Credientials as Envoirnemnt Variables
os.environ['KAGGLE_USERNAME'] = kaggle_dictionary['username']
os.environ['KAGGLE_KEY'] = kaggle_dictionary['key']

In [7]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:00<00:00, 84.3MB/s]



In [8]:
# Unzipping Dataset:
!unzip /content/imdb-dataset-of-50k-movie-reviews.zip

Archive:  /content/imdb-dataset-of-50k-movie-reviews.zip
  inflating: IMDB Dataset.csv        


In [9]:
# Loading Dataset into Pandas DataFrame
movies_dataset = pd.read_csv('/content/IMDB Dataset.csv')

In [10]:
# Pritning 5 rows of Dataset
movies_dataset.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [11]:
# Shape of Dataset
movies_dataset.shape

(50000, 2)

In [12]:
# Coutning Sentiments
movies_dataset['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [ ]:
# Replacing Sentiment values with 0 & 1
movies_dataset.replace({'sentiment': {'positive':1 ,'negative':0}}, inplace=True)

In [14]:
movies_dataset.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


**Splitting data into Training & Test**

In [15]:
train_data, test_data = train_test_split(movies_dataset, test_size=0.2, random_state=42)

In [16]:
print(train_data.shape, test_data.shape)

(40000, 2) (10000, 2)


**Data Preprocessing**

In [28]:
# Tokennizer Text Data
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data['review'])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen=200)

In [29]:
print(X_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [30]:
print(X_test)

[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [31]:
Y_train = train_data['sentiment']
Y_test = test_data['sentiment']

In [32]:
print(Y_train)
print(Y_test)

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64
33553    1
9427     1
199      0
12447    1
39489    0
        ..
28567    0
25079    1
18707    1
15200    0
5857     1
Name: sentiment, Length: 10000, dtype: int64


In [33]:
print("X_train:", X_train.shape)
print("y_train:", Y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", Y_test.shape)

X_train: (40000, 200)
y_train: (40000,)
X_test: (10000, 200)
y_test: (10000,)


**Building LSTM Model**

In [34]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

model = Sequential([
    Input(shape=(200,)),
    Embedding(input_dim=5000, output_dim=128),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

**Training Model**

In [35]:
model.fit(X_train, Y_train, epochs=5,batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 380s 754ms/step - accuracy: 0.7919 - loss: 0.4479 - val_accuracy: 0.7991 - val_loss: 0.4440
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 388s 776ms/step - accuracy: 0.8540 - loss: 0.3520 - val_accuracy: 0.8668 - val_loss: 0.3205
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 382s 764ms/step - accuracy: 0.8621 - loss: 0.3308 - val_accuracy: 0.8634 - val_loss: 0.3504
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 424s 849ms/step - accuracy: 0.8877 - loss: 0.2790 - val_accuracy: 0.8741 - val_loss: 0.3069
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 386s 737ms/step - accuracy: 0.9091 - loss: 0.2297 - val_accuracy: 0.8702 - val_loss: 0.3356


**Model Evaluation**

In [37]:
loss, accuracy = model.evaluate(X_test, Y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy:  {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 53s 169ms/step - accuracy: 0.8775 - loss: 0.3160
Test Loss: 0.3159883916378021
Test Accuracy:  0.8774999976158142


**Predictive System**

In [42]:
def predict_sentiment(review):
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen=200)
  prediction = model.predict(padded_sequence)
  sentiment = "Positive" if prediction[0][0] > 0.5 else "Negative"
  return sentiment

In [43]:
# Example:
new_review = "This movie was Fantastic and very Good"
sentiment = predict_sentiment(new_review)
print(f"The Sentiment of review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
The Sentiment of review is: Positive


In [44]:
# Example:
new_review = "This movie was very bad and story was not good enough"
sentiment = predict_sentiment(new_review)
print(f"The Sentiment of review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
The Sentiment of review is: Negative
